# ResNet Clothing Classifier Training
This notebook is automatically generated to train our ResNet18 model on Kaggle GPUs.
**Instructions:**
1. Add your clothing dataset to this Kaggle Notebook via "Add Input".
2. Update the `DATASET_PATH` variable in the 3rd cell to point to your new dataset folder.
3. Run all cells! The trained weights will be saved to `/kaggle/working/resnet_clothing.pth` so you can download them.


In [ ]:
import torch
import torch.nn as nn
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision import models

def get_clothing_dataloaders(data_dir, batch_size=32):
    # Transforms for ResNet
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print(f"Loading images from {data_dir}...")
    full_dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
    
    # 80/20 split
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, full_dataset.classes

In [ ]:
class ClothingClassifier(nn.Module):
    def __init__(self, num_classes):
        super(ClothingClassifier, self).__init__()
        # Load pre-trained ResNet18
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Modify the final layer
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)
        
    def forward(self, x):
        return self.model(x)

In [ ]:
# TODO: UPDATE THIS PATH to point to your Kaggle Input dataset!
# Example: "/kaggle/input/clothing-dataset-full"
DATASET_PATH = "/kaggle/input/YOUR_DATASET_NAME_HERE"

def train():
    # Automatically select CUDA if available (Kaggle GPU)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}\n")

    print("1. Loading Data...")
    train_loader, test_loader, class_names = get_clothing_dataloaders(DATASET_PATH, batch_size=32)
    num_classes = len(class_names)
    print(f"Found {num_classes} classes!\n")

    print("2. Building Model...")
    model = ClothingClassifier(num_classes=num_classes)
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    epochs = 10 

    print("\n3. Starting Training Loop...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() 
            
        train_loss /= len(train_loader)
        print(f"Epoch {epoch+1:2d}/{epochs} | Training Loss: {train_loss:.4f}")

    print("\n4. Saving the trained model...")
    # Save the weights to Kaggle's working directory so it can be downloaded
    model_save_path = "/kaggle/working/resnet_clothing.pth"
    torch.save(model.state_dict(), model_save_path)
    print(f"Model saved to '{model_save_path}'!")
    
train()